In [1]:
import pandas as pd
import numpy as np

In [2]:
nav = pd.read_csv("data/raw/02_nav_history.csv")

In [3]:
nav.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB


In [4]:
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [5]:
nav.describe()

,amfi_code,nav
count,46000.000000,46000.000000
mean,120247.000000,269.570265
std,14352.317221,577.187060
min,100016.000000,26.136600
25%,118632.750000,69.170425
50%,119551.500000,122.732150
75%,120842.250000,260.338675
max,149324.000000,4268.549700


In [6]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [7]:
nav["date"] = pd.to_datetime(nav["date"])

In [8]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [9]:
nav = nav.drop_duplicates()

In [10]:
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

In [11]:
nav = nav[nav["nav"] > 0]

In [12]:
nav.to_csv(
    "data/processed/clean_nav.csv",
    index=False
)

In [13]:
tx = pd.read_csv(
    "data/raw/08_investor_transactions.csv"
)

In [14]:
tx["transaction_date"] = pd.to_datetime(
    tx["transaction_date"]
)

In [15]:
tx["transaction_type"] = (
    tx["transaction_type"]
    .str.strip()
    .str.title()
)

In [16]:
valid = [
    "Sip",
    "Lumpsum",
    "Redemption"
]

tx = tx[
    tx["transaction_type"].isin(valid)
]

In [17]:
tx = tx[
    tx["amount_inr"] > 0
]

In [18]:
tx = tx[
    tx["kyc_status"].isin(
        ["Verified","Pending"]
    )
]

In [19]:
tx.to_csv(
    "data/processed/clean_transactions.csv",
    index=False
)

In [20]:
perf = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)

In [21]:
numeric = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "expense_ratio_pct",
    "sharpe_ratio"
]

for c in numeric:
    perf[c] = pd.to_numeric(
        perf[c],
        errors="coerce"
    )

In [22]:
perf = perf[
    perf["expense_ratio_pct"]
    .between(0.1,2.5)
]

In [23]:
perf["negative_sharpe"] = (
    perf["sharpe_ratio"] < 0
)

In [24]:
perf.to_csv(
    "data/processed/clean_performance.csv",
    index=False
)